# Long-Form Policy Pipeline

This notebook builds a training table from existing long-form sweeps, trains a policy model, and generates fresh demo outputs chosen by that policy.

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 3.2').exists():
            return path
    raise RuntimeError('Could not resolve repo root.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.2' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import longform_policy_pipeline as lpp
importlib.reload(lpp)
REPO

In [ ]:
cfg = lpp.PolicyConfig(
    n_demo_songs=3,
)
RUN_ALL = True
print(cfg.output_root / cfg.tag)

In [ ]:
examples = lpp.discover_training_examples(cfg)
print('Training examples:', len(examples))
pd.DataFrame(examples).head(10)

In [ ]:
summary = None
if RUN_ALL:
    summary = lpp.run_full_pipeline(cfg)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to run the full policy pipeline.')

In [ ]:
run_dir = cfg.output_root / cfg.tag
training_table = run_dir / 'training_table.csv'
policy_metrics = run_dir / 'policy_metrics.json'
demo_manifest = run_dir / 'demo_manifest.csv'
if training_table.exists():
    display(pd.read_csv(training_table).head(15))
    print(policy_metrics)
    print(demo_manifest)
    if demo_manifest.exists():
        display(pd.read_csv(demo_manifest))
else:
    print('No outputs yet.')